In [1]:
import sqlite3
import os, sys

# === 共通設定 ===
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config import PROJECT_DIR   # ← これを使う

# === パス設定 ===
if os.name == "nt":
    user_base = os.path.join(os.environ["USERPROFILE"], "myenv310", PROJECT_DIR)
else:
    user_base = os.path.join(os.path.expanduser("~"), "myenv310", PROJECT_DIR)

db_path      = os.path.join(user_base, "db", "output.db")
columns_file = os.path.join(user_base, "db", "columns_with_type.txt")
table_name   = "result_table"

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# ✅ 既存カラム取得
cur.execute(f"PRAGMA table_info({table_name});")
existing_cols = {row[1] for row in cur.fetchall()}

# ✅ columns_with_type.txt 読み込み
with open(columns_file, encoding='utf-8') as f:
    cols_types = [line.strip().split() for line in f if line.strip()]

# ✅ 新カラムのみ追加
added = 0
for col, dtype in cols_types:
    if col not in existing_cols:
        alter_sql = f'ALTER TABLE {table_name} ADD COLUMN `{col}` {dtype};'
        cur.execute(alter_sql)
        print(f"[INFO] ✅ 追加: {col} {dtype}")
        added += 1

conn.commit()
conn.close()

if added:
    print(f"[INFO] ✅ 新規カラム {added} 件 追加完了")
else:
    print("[INFO] ✅ 追加カラムなし、既に全て存在")


[INFO] ✅ 追加カラムなし、既に全て存在
